# Function Learning Experiment

**Goal:** Learn polynomial approximations to target functions over $p$-adic fields.
We minimize cross-entropy loss:
$$L(\theta) = -\sum_i [y_i \log(\sigma(f_\theta(x_i))) + (1-y_i) \log(1 - \sigma(f_\theta(x_i)))]$$
where $f_\theta(x) = a_0 + a_1 x + \cdots + a_n x^n$ and $\sigma$ is the sigmoid function.

**Two tasks:**
- **Zero function**: Learn $f(x) = 0$ for all $x$ (trivial solution exists)
- **One function**: Learn $f(x) = 1$ for all $x$ (constant polynomial)

We compare multiple optimization methods: **Greedy**, **Greedy-deg2**, **MCTS**, **DAG-MCTS**, and **UCT**.

In [ ]:
include("../../../src/NAML.jl")
include("../util.jl")

using Oscar
using .NAML
using Printf

## 1. Configuration

Set the experiment parameters here.

In [ ]:
# Field configuration
PRIME = 2
PREC = 20
K = PadicField(PRIME, PREC)

# Problem configuration
POLY_DEGREE = 3          # Degree of polynomial to learn
N_POINTS = 5             # Number of random test points
TARGET_FN = "zero"       # Target function: "zero" or "one"
THRESHOLD = 0.5          # Sigmoid threshold
SCALE = 1.0              # Sigmoid scale parameter
N_EPOCHS = 20            # Optimization epochs per optimizer

println("Configuration:")
println("  Prime: $PRIME")
println("  Precision: $PREC")
println("  Polynomial degree: $POLY_DEGREE")
println("  Test points: $N_POINTS")
println("  Target function: $TARGET_FN")
println("  Threshold: $THRESHOLD, Scale: $SCALE")
println("  Epochs: $N_EPOCHS")

## 2. Generate Test Data

Generate random $p$-adic test points with target labels.

In [ ]:
# Generate random test points
x_values = [generate_random_padic(PRIME, PREC, 0, 8) for _ in 1:N_POINTS]

# Target values based on target function
if TARGET_FN == "zero"
    y_values = [0.0 for _ in 1:N_POINTS]
elseif TARGET_FN == "one"
    y_values = [1.0 for _ in 1:N_POINTS]
else
    error("Unknown target function: $TARGET_FN")
end

data = collect(zip(x_values, y_values))

println("Test data ($(length(data)) points):")
for (i, (x, y)) in enumerate(data)
    println("  $i. x = $x, y = $y")
    println("     |x| = $(Float64(abs(x)))")
end

## 3. Create Loss Function

Create cross-entropy loss with sigmoid activation for smooth optimization.

In [ ]:
# Setup polynomial ring
var_names = ["x"]
param_names = ["a$i" for i in 0:POLY_DEGREE]
all_vars = vcat(var_names, param_names)
R, vars = polynomial_ring(K, all_vars)

x = vars[1]
params = vars[2:end]

# Create polynomial: a0 + a1*x + a2*x^2 + ... + an*x^n
poly_expr = sum(params[i] * x^(i-1) for i in 1:length(params))

println("Polynomial structure: f(x) = a0 + a1*x + a2*x^2 + ... + a$(POLY_DEGREE)*x^$(POLY_DEGREE)")

# Setup model
param_info = vcat([true], [false for _ in 1:POLY_DEGREE+1])  # x is data, params are parameters
fun = NAML.AbsolutePolynomialSum([poly_expr])
model = NAML.AbstractModel(fun, param_info)

# Create cross-entropy loss
specialized_models = [NAML.specialise(model, [val]) for (val, _) in data]
batch_evals = [NAML.batch_evaluate_init(specialized_models[i]) for i in eachindex(specialized_models)]

function eval_fn(params::Vector{NAML.ValuationPolydisc{S,T,N}}) where {S, T, N}
    return [begin
        loss = 0.0
        for i in eachindex(data)
            val_float = Float64(batch_evals[i](param))
            y = y_values[i]

            # Cross-entropy: -[y*log(p) + (1-y)*log(1-p)]
            # where p = sigmoid((val - threshold)/scale)
            z = (val_float - THRESHOLD) / SCALE
            prob = 1.0 / (1.0 + exp(-z))

            # Clip probabilities to avoid log(0)
            prob = max(min(prob, 0.9999), 0.0001)

            if y > 0.5  # y = 1
                loss += -log(prob)
            else  # y = 0
                loss += -log(1 - prob)
            end
        end
        loss
    end for param in params]
end

function grad_fn(vs::Vector{NAML.ValuationTangent{S,T,N}}) where {S, T, N}
    return [0.0 for _ in vs]
end

loss = NAML.Loss(eval_fn, grad_fn)

# Initialize parameters at origin
param_center = [K(0) for _ in 1:POLY_DEGREE+1]
initial_param = NAML.ValuationPolydisc(param_center, [0 for _ in 1:POLY_DEGREE+1])
initial_loss = loss.eval([initial_param])[1]

println("\nInitial parameter: center at origin, radius 0")
println("Initial loss: $initial_loss")

## 4. Define Optimizers

We compare several optimization strategies with different hyperparameters.

In [ ]:
optimizer_defs = [
    (name="Greedy",
     init=(p, l) -> NAML.greedy_descent_init(p, l, 1, (false, 1))),
    
    (name="Greedy-deg2",
     init=(p, l) -> NAML.greedy_descent_init(p, l, 1, (false, 2))),
    
    (name="MCTS-50",
     init=(p, l) -> NAML.mcts_descent_init(p, l,
         NAML.MCTSConfig(num_simulations=50, exploration_constant=1.41,
                         selection_mode=NAML.BestValue, degree=1))),
    
    (name="MCTS-100",
     init=(p, l) -> NAML.mcts_descent_init(p, l,
         NAML.MCTSConfig(num_simulations=100, exploration_constant=1.41,
                         selection_mode=NAML.BestValue, degree=1))),
    
    (name="DAG-MCTS-100",
     init=(p, l) -> NAML.dag_mcts_descent_init(p, l,
         NAML.DAGMCTSConfig(num_simulations=100, exploration_constant=1.41,
                            degree=1, persist_table=true,
                            selection_mode=NAML.BestValue))),
    
    (name="UCT",
     init=(p, l) -> NAML.uct_descent_init(p, l,
         NAML.UCTConfig(max_depth=10, num_simulations=100,
                        exploration_constant=1.41, degree=1))),
]

println("Optimizers to compare:")
for od in optimizer_defs
    println("  - $(od.name)")
end

## 5. Run Experiments

Run each optimizer and record loss trajectories.

In [ ]:
# Store results: name => (losses, time, final_param)
results = Dict{String, Any}()

for od in optimizer_defs
    println("\nRunning $(od.name)...")
    
    optim = od.init(initial_param, loss)
    losses = Float64[]
    
    t_start = time()
    for epoch in 1:N_EPOCHS
        current_loss = NAML.eval_loss(optim)
        push!(losses, current_loss)
        NAML.step!(optim)
    end
    final_loss = NAML.eval_loss(optim)
    push!(losses, final_loss)
    elapsed = time() - t_start
    
    results[od.name] = Dict(
        "losses" => losses,
        "time" => elapsed,
        "final_loss" => final_loss,
        "final_param" => optim.param,
    )
    
    improvement = (initial_loss > 0) ? (initial_loss - final_loss) / initial_loss * 100 : 0.0
    @printf("  %s: final_loss = %.6e, improvement = %.1f%%, time = %.2fs\n",
            od.name, final_loss, improvement, elapsed)
end

println("\nAll optimizers finished.")

## 6. Summary Table

In [ ]:
println("\nSUMMARY")
println("="^70)
println("Problem: $(PRIME)-adic, degree $POLY_DEGREE, $N_POINTS points, target: $TARGET_FN")
println("Initial loss: $initial_loss")
println()
@printf("%-15s %15s %12s %12s\n", "Optimizer", "Final Loss", "Improv %", "Time (s)")
println("-"^55)

for od in optimizer_defs
    r = results[od.name]
    improv = (initial_loss > 0) ? (initial_loss - r["final_loss"]) / initial_loss * 100 : 0.0
    @printf("%-15s %15.6e %11.1f%% %12.2f\n",
            od.name, r["final_loss"], improv, r["time"])
end

# Find best
best_name = ""
best_loss = Inf
for od in optimizer_defs
    if results[od.name]["final_loss"] < best_loss
        best_loss = results[od.name]["final_loss"]
        best_name = od.name
    end
end
println("\nBest optimizer: $best_name (loss = $best_loss)")

## 7. Loss Trajectories

Display the loss over epochs for each optimizer.

In [ ]:
# Print loss trajectories as text
println("\nLoss trajectories (selected epochs):")
println()

# Header
header = @sprintf("%-8s", "Epoch")
for od in optimizer_defs
    header *= @sprintf(" %15s", od.name)
end
println(header)
println("-"^(8 + 16 * length(optimizer_defs)))

# Print selected epochs
epochs_to_show = unique(vcat([1, 2, 3, 5, 10], collect(10:10:N_EPOCHS), [N_EPOCHS, N_EPOCHS+1]))
epochs_to_show = filter(e -> e <= N_EPOCHS + 1, sort(epochs_to_show))

for epoch in epochs_to_show
    label = epoch <= N_EPOCHS ? string(epoch) : "final"
    row = @sprintf("%-8s", label)
    for od in optimizer_defs
        losses = results[od.name]["losses"]
        if epoch <= length(losses)
            row *= @sprintf(" %15.6e", losses[epoch])
        end
    end
    println(row)
end

## 8. Learned Coefficients

Inspect the polynomial coefficients found by the best optimizer.

In [ ]:
println("Learned coefficients ($best_name):")
println()

best_param = results[best_name]["final_param"]
best_center = NAML.center(best_param)
best_radius = NAML.radius(best_param)

for (i, coef) in enumerate(best_center)
    deg = i - 1
    coef_abs = Float64(abs(coef))
    r = best_radius[i]
    @printf("  a%d (x^%d): |a%d| = %.6e, radius = %d\n", deg, deg, deg, coef_abs, r)
end

# Verify: evaluate learned polynomial on test data
println("\nVerification on test data:")
for (i, (x, y)) in enumerate(data)
    f_x = sum(best_center[k] * x^(k-1) for k in 1:length(best_center))
    f_x_float = Float64(abs(f_x))
    
    # Compute predicted probability
    z = (f_x_float - THRESHOLD) / SCALE
    prob = 1.0 / (1.0 + exp(-z))
    
    @printf("  Point %d: f(x) = %.6e, prob = %.4f, target = %.0f\n", i, f_x_float, prob, y)
end

## 9. Compare Zero vs One Function

Run both tasks to compare difficulty.

In [ ]:
comparison_configs = [
    (target="zero", degree=3, points=5),
    (target="one", degree=3, points=5),
    (target="zero", degree=4, points=6),
    (target="one", degree=4, points=6),
]

COMP_EPOCHS = 15

println("\nCOMPARISON: Zero vs One Function")
println("="^90)

# Use a subset of optimizers for speed
comp_optimizers = [(name="Greedy", init=(p, l) -> NAML.greedy_descent_init(p, l, 1, (false, 1))),
                   (name="Greedy-deg2", init=(p, l) -> NAML.greedy_descent_init(p, l, 1, (false, 2))),
                   (name="MCTS-100", init=(p, l) -> NAML.mcts_descent_init(p, l,
                       NAML.MCTSConfig(num_simulations=100, exploration_constant=1.41,
                                      selection_mode=NAML.BestValue, degree=1)))]

for cfg in comparison_configs
    println("\nTarget: $(cfg.target), Degree: $(cfg.degree), Points: $(cfg.points)")
    
    # Generate test data
    xs = [generate_random_padic(PRIME, PREC, 0, 8) for _ in 1:cfg.points]
    ys = cfg.target == "zero" ? [0.0 for _ in 1:cfg.points] : [1.0 for _ in 1:cfg.points]
    d = collect(zip(xs, ys))
    
    # Create loss
    var_names = ["x"]
    param_names = ["a$i" for i in 0:cfg.degree]
    all_vars = vcat(var_names, param_names)
    R, vars = polynomial_ring(K, all_vars)
    x = vars[1]
    params = vars[2:end]
    poly_expr = sum(params[i] * x^(i-1) for i in 1:length(params))
    
    param_info = vcat([true], [false for _ in 1:cfg.degree+1])
    fun = NAML.AbsolutePolynomialSum([poly_expr])
    model = NAML.AbstractModel(fun, param_info)
    
    specialized_models = [NAML.specialise(model, [val]) for (val, _) in d]
    batch_evals = [NAML.batch_evaluate_init(specialized_models[i]) for i in eachindex(specialized_models)]
    
    function eval_fn(params::Vector{NAML.ValuationPolydisc{S,T,N}}) where {S, T, N}
        return [begin
            loss = 0.0
            for i in eachindex(d)
                val_float = Float64(batch_evals[i](param))
                y = ys[i]
                z = (val_float - THRESHOLD) / SCALE
                prob = 1.0 / (1.0 + exp(-z))
                prob = max(min(prob, 0.9999), 0.0001)
                if y > 0.5
                    loss += -log(prob)
                else
                    loss += -log(1 - prob)
                end
            end
            loss
        end for param in params]
    end
    
    grad_fn(vs) = [0.0 for _ in vs]
    l = NAML.Loss(eval_fn, grad_fn)
    
    param_center = [K(0) for _ in 1:cfg.degree+1]
    p0 = NAML.ValuationPolydisc(param_center, [0 for _ in 1:cfg.degree+1])
    l0 = l.eval([p0])[1]
    
    for od in comp_optimizers
        opt = od.init(p0, l)
        for e in 1:COMP_EPOCHS
            NAML.step!(opt)
        end
        fl = NAML.eval_loss(opt)
        improv = (l0 > 0) ? (l0 - fl) / l0 * 100 : 0.0
        @printf("  %-15s final=%.6e  improv=%.1f%%\n", od.name, fl, improv)
    end
end

## Summary

This notebook demonstrates learning constant target functions over $p$-adic fields using
polynomial approximations and cross-entropy loss.

**Key findings:**
- The zero function is typically easier to learn (trivial solution: all coefficients = 0)
- The one function requires finding appropriate constant or near-constant polynomials
- Greedy with higher degree (deg2) can explore more aggressively
- MCTS methods provide more thorough exploration at higher computational cost

**To reproduce at scale:** Use `run_experiments.jl` with `--save` to generate JSON results,
then `generate_tables.jl` to produce a unified LaTeX document with all tables.